In [8]:
!pip install groq
!pip install dotenv


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\beimn\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\beimn\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [15]:
#Set Groq API token
import os
from dotenv import load_dotenv, find_dotenv

# Find .env starting from the notebook's folder and load GROQ_API_KEY into os.environ
# override=True picks up edits to .env without restarting the kernel
load_dotenv(find_dotenv(usecwd=True), override=True)

if not os.getenv("GROQ_API_KEY"):
    raise RuntimeError("GROQ_API_KEY not found - check the .env file next to this notebook")
print("GROQ_API_KEY loaded")




GROQ_API_KEY loaded


## Define Tools

In [16]:
from geopy.geocoders import Nominatim
import requests

def get_weather(city: str) -> int:
    """ Provide weather information with city as input and temperature as output in degree celsius(integer) """

    #api call

    try:
        # convert city name to latitude and longitude
        geo_locator = Nominatim(user_agent="weather_agent_colab")
        location = geo_locator.geocode(city)
        if not location:
            raise ValueError("City not found")

        lat, lon = location.latitude, location.longitude

        # fetch weather data from open-meteo free API
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # extract temperature
        temperature = int(round(data["current_weather"]["temperature"]))
        return temperature
        
    except Exception as e:
        print(f"Error fetching weather: {e}")
        return 0

In [17]:
# Suggest acctivity 
def suggest_activity(temperature: int) -> str:
    "Suggests activity to do as per temperature as input"
    if temperature < 21:
        return "Eat Bokolo"
    elif temperature > 20 and temperature < 25:
        return "Dance with your girl"
    else:
        return "Jump on bed"

In [19]:
# Create all TOOLS
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "weather",
            "description": (
                "Allows to get weather of a city in degree celsius"
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city",
                    }
                },
                "required": ["city"]
            },
        },
    },
    
    {
        "type": "function",
        "function": {
            "name": "suggest_activity",
            "description": (
                "Suggests activity to do as per temperature as input"
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "temperature": {
                        "type": "integer",
                        "description": "Temperature of the city",
                    }
                },
                "required": ["temperature"]
            },
        },
    }
]

In [13]:
# Tool functions - maps strings to actual functions
TOOL_FUNCTION = {
    "weather": get_weather,
    "suggest_activity": suggest_activity
}

In [ ]:
import json

QUESTION = "suggest me an activity based on the weather in Addis Ababa"

# Create messages
MESSAGES = [
    {
        "role": "user", "content": QUESTION
    }
]

MAX_STEPS = 5

for step in range(MAX_STEPS):
    
    # Call LLM with TOOL
    from groq import Groq
    client = Groq()
    MODEL = "openai/gpt-oss-120b"

    res = client.chat.completions.create(
        model = MODEL,
        max_tokens = 1000,
        tools = TOOLS,
        messages = MESSAGES
    )
    
    # Append Assistant Message
    MESSAGES.append(res.choices[0].message)
    
    if not res.choices[0].message.tool_calls:
        print(f"final Output: {res.choices[0].message.content}")
        break
    
    # Execute the tool to send back observation to model
    for call in res.choices[0].message.tool_calls:
        tool_id =call.id
        func_name =call.function.name
        _args = json.loads(call.function.arguments)

        tool_output = TOOL_FUNCTION[func_name](**_args)
        MESSAGES.append(
            {
                "role": "tool", "content": str(tool_output), "tool_call_id": tool_id, "name": func_name
            }
        )
        print(f"Tool call -> ",  tool_id, func_name, _args, tool_output)

BadRequestError: Error code: 400 - {'error': {'message': 'invalid JSON schema for tool suggest_activity, tools[1].function.parameters: "file:///home/di/params.json#" is not valid against metaschema: jsonschema validation failed with \'https://json-schema.org/draft/2020-12/schema#\'\n- at \'\': \'allOf\' failed\n  - at \'/properties/city\': \'allOf\' failed\n    - at \'/properties/city/type\': \'anyOf\' failed\n      - at \'/properties/city/type\': value must be one of \'array\', \'boolean\', \'integer\', \'null\', \'number\', \'object\', \'string\'\n      - at \'/properties/city/type\': got string, want array', 'type': 'invalid_request_error', 'param': 'tool suggest_activity, tools[1].function.parameters'}}